In [92]:
import numpy as np
from numpy.typing import NDArray
import pandas as pd

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBRegressor
import lightgbm as lgb

import os
import sys
root_path="/home/iaw/DATA2/AAReact/src"
sys.path.append(root_path)
from util.RegressMetrics import r2_score, mse_score, mae_score, rmse_score
from util.train_tools import build_model, search_parms, split_data, load_data, eval_dataset_split
from tool.analysis import draw_pred_result

import seaborn as sns
from matplotlib import pyplot as plt

from typing import List, Tuple, Dict
from rich.table import Table
from rich import print as rp
from rich.progress import track
import shap
import pickle

from joblib import dump, load
from copy import deepcopy

from joblib import dump

In [ ]:
seed = 1
test_size = 0.2
# 超参之后的参数
xgb_best_params = {
      "colsample_bytree": 0.9
    , "learning_rate": 0.1
    , "max_depth": 8
    , "min_child_weight": 5
    , "n_estimators": 50
    , "reg_alpha": 0.5
    , "reg_lambda": 1.5
    , "subsample": 0.8
}


In [ ]:
# 模型初始训练数据
prefix1 = "/home/iaw/DATA2/AAReact/DataSet/Data_All/4_data_for_train/rdkit_xtb/train_test"
data_x, data_y, x_label, data_class = load_data(
    "{}_data_x.npy".format(prefix1), 
    "{}_data_y.npy".format(prefix1), 
    "{}_x_label.pkl".format(prefix1), 
    "{}_data_class.pkl".format(prefix1)
)


In [96]:
def del_tail8(data_x: NDArray
              , data_y: NDArray
              , x_label: Dict[str, str]
              , data_class: List[int]
              , data_name: List[str]
              , data_batch: List[int]):

    return (data_x[:-8, :]
          , data_y[:-8]
          , x_label
          , data_class[:-8]
          , data_name[:-8]
          , data_batch[:-8])


In [98]:
# 数据集外验证集
prefix2 = "/home/iaw/DATA2/AAReact/DataSet/Data_All/4_data_for_train/rdkit_xtb/extra"
data_x_new, data_y_new, x_label_new, data_class_new, data_name_new, data_batch_new = load_data(
    "{}_data_x.npy".format(prefix2), 
    "{}_data_y.npy".format(prefix2), 
    "{}_x_label.pkl".format(prefix2), 
    "{}_data_class.pkl".format(prefix2),
    "{}_data_name.pkl".format(prefix2),
    "{}_data_batch.pkl".format(prefix2)
)
data_x_new, data_y_new, x_label_new, data_class_new, data_name_new, data_batch_new = del_tail8(data_x_new
                                                                                               , data_y_new
                                                                                               , x_label_new
                                                                                               , data_class_new
                                                                                               , data_name_new
                                                                                               , data_batch_new)
## 这里需要对data_batch进行一次矫正
#- extra 一共28个数据点
#- 前6个是一组, 紧接着的6个也是一组, 随后的8个为两组, 均是CAT-71对应的不同的温度, 然后是4个CAT-72的, 4个CAT-73的
correction_batch = [
    # 第1组：6个，编号 3 CAT-41 new_rea
    3, 3, 3, 3, 3, 3,    
    # 第2组：6个，编号 4 CAT-42 new_rea
    4, 4, 4, 4, 4, 4,
    # 第3组：8个（两组），编号 5、6 CAT-71 TEMP
    5, 5, 5, 5, 6, 6, 6, 6,
    # 第4组：4个，CAT-72，编号 7    CAT-72
    #7, 7, 7, 7,
    # 第5组：4个，CAT-73，编号 8    CAT-73
    #8, 8, 8, 8
]
assert len(correction_batch) == len(data_batch_new), "Error[iaw]: len(correction_batch) = len(data_batch_new)"
data_batch_new = correction_batch

In [ ]:
# 从数据集中去除7 8

In [105]:
with open("add_data_result.csv", "w+") as F:
    F.writelines("AddDataBatch,test0,extra_test0,test1,extra_test1\n")
    for add_data_batch in [5, 6]:  
        add_data_idx = []
        extra_test_data_idx = []
        for i, i_t in enumerate(data_batch_new):
            if i_t == add_data_batch:
                add_data_idx.append(i)
            else:
                extra_test_data_idx.append(i)


        data_x_iter0 = deepcopy(data_x)
        data_y_iter0 = deepcopy(data_y)
        data_class_iter0 = deepcopy(data_class)

        X_train_iter0, X_test_iter0, y_train_iter0, y_test_iter0,  class_train_iter0, class_test_iter0 = train_test_split(
            data_x_iter0,        
            data_y_iter0,
            data_class_iter0,
            test_size=test_size,
            random_state=seed, 
        )

        # old tain + new train
        X_train_iter1 = np.vstack([X_train_iter0, data_x_new[add_data_idx,:]])
        y_train_iter1 = np.hstack([y_train_iter0, data_y_new[add_data_idx]])
        class_train_iter1 = deepcopy(class_train_iter0)
        for i in add_data_idx:
            class_train_iter1.append(data_class_new[i])

        # 抛开额外新加入train的数据, 作为extra_test
        extra_X_test_iter0 = deepcopy(data_x_new[extra_test_data_idx,:])
        extra_y_test_iter0 = deepcopy(data_y_new[extra_test_data_idx])
        extra_class_test_iter0 = []
        for i in extra_test_data_idx:
            extra_class_test_iter0.append(data_class_new[i])

        # 加载训练好的模型
        model_iter0 = load("/home/iaw/DATA2/AAReact/train/output/pt/xgb_rdkit_xtb_seed_0-1_test_0-2.pkl")

        # 训练新的模型
        model_iter1 = build_model("xgb", seed, n_cpu = -1)
        model_iter1.set_params(**xgb_best_params)
        model_iter1.fit(X_train_iter1, y_train_iter1)
        # 在本地保存
        dump(model_iter1, "./pt/iter1_add-data-batch_{}.pkl".format(add_data_batch))

        # predict -> test0
        test_pred_iter0 = model_iter0.predict(X_test_iter0)
        extra_test_pred_iter0 = model_iter0.predict(extra_X_test_iter0)

        # predict -> test1
        test_pred_iter1 = model_iter1.predict(X_test_iter0)
        extra_test_pred_iter1 = model_iter1.predict(extra_X_test_iter0)

        F.writelines("{},{:.4f},{:.4f},{:.4f},{:.4f}\n".format(add_data_batch
              , r2_score(y_pred = test_pred_iter0, y_true = y_test_iter0)
              , r2_score(y_pred = extra_test_pred_iter0, y_true = extra_y_test_iter0)
              , r2_score(y_pred = test_pred_iter1, y_true = y_test_iter0)
              , r2_score(y_pred = extra_test_pred_iter1, y_true = extra_y_test_iter0)))

In [107]:
import numpy as np
import matplotlib.pyplot as plt

# ====================== 数据 ======================
batch = [5, 6]
test0      = [0.8387, 0.8387]
extra_test0= [0.7632, 0.8407]
test1      = [0.8175, 0.8113]
extra_test1= [0.9514, 0.9726]

labels = ["add extra group1", "add extra group2"]
n = len(labels)
x = np.arange(n)
width = 0.2  # 4组柱子，窄一点更美观

# ====================== 完全对齐你的 SCI 样式 ======================
plt.rcParams.update({
    'font.family': 'Times New Roman',
    'font.size': 11,
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'axes.linewidth': 1.2,
    'figure.dpi': 300,
    'savefig.dpi': 600,
    'savefig.bbox': 'tight',
    'savefig.format': 'pdf'
})

# 配色（和你论文风格一致：沉稳、学术、低饱和）
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']  # 蓝、橙、绿、红
bar_names = ['Test', 'Test[iter]', 'Extra Test', 'Extra Test[iter]']

# ====================== 绘图 ======================
fig, ax = plt.subplots(figsize=(7, 5))

# 画 4 组柱子
offsets = [-1.5*width, -0.5*width, 0.5*width, 1.5*width]
data_list = [test0, test1, extra_test0, extra_test1]

for i, (d, color, offset) in enumerate(zip(data_list, colors, offsets)):
    ax.bar(x + offset, d, width, 
           color=color, alpha=0.9, 
           edgecolor='black', linewidth=0.7, 
           label=bar_names[i])

# ====================== 样式（100% 对齐你的代码） ======================
ax.set_ylabel('$R^2$', fontweight='bold')
ax.set_xlabel('Additional Data Group', fontweight='bold')
#ax.set_title('Performance Improvement with Active Learning Data', fontweight='bold', pad=10)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0.70, 1.00)
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.legend(frameon=True, edgecolor='black', fancybox=False)

# ====================== 保存 ======================
plt.tight_layout()
plt.savefig('add_data_effect.png', dpi=600)
plt.close()